<a href="https://colab.research.google.com/github/desmond-lartey/Knowledge-Management-Informatics/blob/Fires/docs/examples/plannerOLD_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 %pip install geoai-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.0/631.0 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.2/882.2 kB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.0/605.0 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
import geopandas as gpd
import pandas as pd
import rasterio, rasterio.mask
from shapely.geometry import box
from pathlib import Path

# --- paths (edit if needed)
TEST_RASTER = "naip_test.tif"                          # NAIP, 4 bands [R,G,B,NIR]
PRED_VECTOR = "naip_test_semantic_prediction.geojson"  # from your vectorize step
OUT_DIR = Path("geoai_diagnostics")
OUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Load predicted polygons
bld = gpd.read_file(PRED_VECTOR)
if "area_m2" not in bld.columns:
    bld["area_m2"] = bld.geometry.area

# Optional: filter tiny artifacts (keep your threshold if you already did)
bld = bld[bld["area_m2"] > 50].copy()
bld = bld.reset_index(drop=True)
print(f"Predicted buildings kept: {len(bld)}")


Predicted buildings kept: 616


In [ ]:
CELL_M = 200  # grid cell size in meters (tunable)

with rasterio.open(TEST_RASTER) as src:
    crs = src.crs
    bounds = src.bounds

xmin, ymin, xmax, ymax = bounds.left, bounds.bottom, bounds.right, bounds.top
xs = np.arange(xmin, xmax, CELL_M)
ys = np.arange(ymin, ymax, CELL_M)

cells = [box(x0, y0, x0 + CELL_M, y0 + CELL_M) for x0 in xs for y0 in ys]
grid = gpd.GeoDataFrame({"cell_id": range(len(cells))}, geometry=cells, crs=crs)
grid_area_m2 = CELL_M * CELL_M
grid.head()


,cell_id,geometry
0,0,"POLYGON ((454841.6 5276774.4, 454841.6 5276974..."
1,1,"POLYGON ((454841.6 5276974.4, 454841.6 5277174..."
2,2,"POLYGON ((454841.6 5277174.4, 454841.6 5277374..."
3,3,"POLYGON ((454841.6 5277374.4, 454841.6 5277574..."
4,4,"POLYGON ((454841.6 5277574.4, 454841.6 5277774..."


In [ ]:
def ndvi_mean_for_geom(geom):
    with rasterio.open(TEST_RASTER) as src:
        try:
            out, _ = rasterio.mask.mask(src, [geom.__geo_interface__], crop=True)
            # NAIP typical order: [R,G,B,NIR]
            R = out[0].astype(np.float32)
            N = out[3].astype(np.float32)
            mask = (R + N) > 0
            if mask.sum() == 0:
                return np.nan
            ndvi = (N - R) / (N + R + 1e-6)
            return float(np.nanmean(np.where(mask, ndvi, np.nan)))
        except Exception:
            return np.nan

ndvi_vals = [ndvi_mean_for_geom(g) for g in grid.geometry]
grid["ndvi_mean"] = ndvi_vals
# quick “green ratio” proxy: 1 if a cell is generally green, else 0 (tune threshold)
grid["green_flag"] = (grid["ndvi_mean"] > 0.3).astype(float)


In [ ]:
# Spatial join for count and size stats
sj = gpd.sjoin(bld[["geometry","area_m2"]], grid[["cell_id","geometry"]], predicate="intersects")
agg = sj.groupby("cell_id").agg(
    bld_count=("area_m2", "size"),
    bld_area_sum=("area_m2", "sum"),
    mean_bld_area=("area_m2", "mean"),
    med_bld_area=("area_m2", "median")
)

grid = grid.join(agg, on="cell_id")
grid[["bld_count","bld_area_sum","mean_bld_area","med_bld_area"]] = \
    grid[["bld_count","bld_area_sum","mean_bld_area","med_bld_area"]].fillna(0)

grid["bld_coverage"] = grid["bld_area_sum"] / grid_area_m2  # fraction of cell covered by buildings

# Optional compactness/fragmentation proxy: many small buildings → fragmented fabric
grid["pct_small_structures"] = 0.0
if len(sj) > 0:
    small = sj[sj["area_m2"] < 60].groupby("cell_id").size()
    grid.loc[small.index, "pct_small_structures"] = (small / grid["bld_count"]).fillna(0).loc[small.index]


In [ ]:
def minmax(s):
    if np.nanmax(s) == np.nanmin(s):
        return s*0
    return (s - np.nanmin(s)) / (np.nanmax(s) - np.nanmin(s))

# helpers, normalized
grid["_n_bld"]     = minmax(grid["bld_count"])
grid["_small"]     = minmax(grid["pct_small_structures"])
grid["_cov"]       = minmax(grid["bld_coverage"])
grid["_inv_ndvi"]  = 1 - minmax(grid["ndvi_mean"].fillna(0))

# Sprawl score: many buildings, many small ones, moderate/low coverage (leapfrog/fragmentation)
grid["sprawl_score"] = (0.45*grid["_n_bld"] + 0.45*grid["_small"] + 0.10*(1 - grid["_cov"])).clip(0,1)

# Environmental degradation: low NDVI + high building coverage (as an impervious proxy)
grid["envdeg_score"] = (0.6*grid["_inv_ndvi"] + 0.4*grid["_cov"]).clip(0,1)


In [ ]:
def recommend(row):
    recs = []

    # Sprawl + low greenness → consolidation + green protection
    if row["sprawl_score"] >= 0.6 and row["ndvi_mean"] < 0.3:
        recs += [
            "Infill & gentle densification near services (form-based code).",
            "Protect peri-urban green wedges; set urban growth boundaries.",
            "Complete sidewalks + crossings; create mixed-use nodes."
        ]
    elif row["sprawl_score"] >= 0.6:
        recs += [
            "Cluster development; curb leapfrog subdivisions.",
            "Improve block connectivity; add missing street links."
        ]

    # Environmental degradation → blue-green retrofits
    if row["envdeg_score"] >= 0.6 and row["ndvi_mean"] < 0.3:
        recs += [
            "Street-tree program & pocket parks for cooling and amenity.",
            "Bioswales / rain gardens; permeable parking retrofits.",
            "Prioritize grey-to-green conversions on public land."
        ]

    # Always suggest low-regret monitoring where signals are mild
    if not recs:
        return "Monitor; no immediate action (track NDVI and coverage quarterly)."
    # de-duplicate while keeping order
    seen, out = set(), []
    for r in recs:
        if r not in seen:
            out.append(r); seen.add(r)
    return "; ".join(out)

grid["recommendations"] = grid.apply(recommend, axis=1)


In [ ]:
gj = OUT_DIR / "urban_diagnostics.geojson"
csv = OUT_DIR / "urban_diagnostics.csv"

cols = ["cell_id","sprawl_score","envdeg_score","bld_count","bld_coverage",
        "mean_bld_area","med_bld_area","pct_small_structures","ndvi_mean","recommendations"]

grid.to_file(gj, driver="GeoJSON")
grid[cols].to_csv(csv, index=False)

print(" Exports written:")
print(" •", gj)
print(" •", csv)


 Exports written:
 • geoai_diagnostics/urban_diagnostics.geojson
 • geoai_diagnostics/urban_diagnostics.csv


In [ ]:
import leafmap
m = leafmap.Map()
m.add_raster(TEST_RASTER, layer_name="NAIP")
m.add_geojson(str(OUT_DIR/"urban_diagnostics.geojson"), layer_name="Diagnostics grid")
m


Map(center=[47.6464835, -117.59043650000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom…

In [ ]:
pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 990.1 kB/s eta 0:00:00


In [ ]:
import os
print(os.getcwd())        # Shows your current folder
print(os.listdir())       # Lists all files in it


/content/drive/MyDrive/planner
['train_segmentation_model.ipynb', 'geoai_diagnostics', 'naip_rgb_train.tif', 'naip_train_buildings.geojson', 'naip_test.tif', 'buildings', 'naip_test_semantic_prediction.tif', 'naip_test_probability_map.tif', 'naip_test_semantic_prediction_threshold2.tif', 'naip_test_semantic_prediction.geojson']


In [ ]:
import osmnx as ox
import rasterio
import geopandas as gpd
from shapely.geometry import box

print(" Using OSMnx version:", ox.__version__)

# --- Load raster and get bounds ---
TEST_RASTER = "/content/drive/MyDrive/planner/naip_test.tif"

with rasterio.open(TEST_RASTER) as src:
    bounds = src.bounds
    crs = src.crs

print(f"Raster CRS: {crs}")

# Create a bounding box polygon in the raster's CRS
bbox_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
gdf_bbox = gpd.GeoDataFrame(geometry=[bbox_geom], crs=crs)

#  Reproject to EPSG:4326 (WGS84)
gdf_bbox = gdf_bbox.to_crs(epsg=4326)
lon_min, lat_min, lon_max, lat_max = gdf_bbox.total_bounds

print(f"Reprojected bbox (lat/lon): {lat_min}, {lat_max}, {lon_min}, {lon_max}")

# --- Use OSMnx with corrected coordinates ---
from osmnx.graph import graph_from_bbox
from osmnx import graph_to_gdfs

# graph_from_bbox expects (north, south, east, west)
bbox = (lat_max, lat_min, lon_max, lon_min)
G = graph_from_bbox(bbox=bbox, network_type="drive")

# Convert to GeoDataFrames
edges, nodes = graph_to_gdfs(G)

print(f" Downloaded {len(edges)} road segments and {len(nodes)} intersections")


 Using OSMnx version: 2.0.6
Raster CRS: EPSG:26911
Reprojected bbox (lat/lon): 47.64277828744407, 47.650188919438484, -117.6039857669145, -117.57688661080043
 Downloaded 3 road segments and 2 intersections


In [ ]:
pip install rasterstats

In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from rasterstats import zonal_stats
from shapely.geometry import box

# --- Define raster and road data ---
SEGMENTATION_RASTER = "/content/drive/MyDrive/planner/naip_test_semantic_prediction.tif"

# Create a uniform analysis grid over your raster
with rasterio.open(SEGMENTATION_RASTER) as src:
    bounds = src.bounds
    transform = src.transform
    res = src.res[0] * 50  # grid ~50× pixel spacing (~50x50 m if 1m resolution)

xmin, ymin, xmax, ymax = bounds.left, bounds.bottom, bounds.right, bounds.top

grid_cells = []
for x0 in np.arange(xmin, xmax, res):
    for y0 in np.arange(ymin, ymax, res):
        x1 = x0 + res
        y1 = y0 + res
        grid_cells.append(box(x0, y0, x1, y1))
grid = gpd.GeoDataFrame(geometry=grid_cells, crs=src.crs)

# --- Building coverage proxy (from segmentation) ---
# assuming building class = 1, vegetation = 2, road = 3
with rasterio.open(SEGMENTATION_RASTER) as src:
    seg = src.read(1)

def ratio_mask(array, class_id):
    return np.count_nonzero(array == class_id) / array.size if array.size > 0 else 0

stats = []
for geom in grid.geometry:
    try:
        out_image, out_transform = rasterio.mask.mask(src, [geom], crop=True)
        arr = out_image[0]
        stats.append({
            "bld_coverage": ratio_mask(arr, 1),
            "veg_coverage": ratio_mask(arr, 2)
        })
    except:
        stats.append({"bld_coverage": 0, "veg_coverage": 0})

grid = pd.concat([grid, pd.DataFrame(stats)], axis=1)

# --- Compute road density ---
edges = edges.to_crs(grid.crs)
grid["road_density_km_per_km2"] = grid.geometry.apply(
    lambda g: edges.clip(g).length.sum() / 1000 / (g.area / 1e6)
)

# --- Derive proxy indicators ---
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

grid["greenness"] = minmax(grid["veg_coverage"])
grid["openness"] = 1 - grid["bld_coverage"]
grid["enclosure"] = grid["bld_coverage"]
grid["walkability"] = (minmax(grid["road_density_km_per_km2"]) * (1 - minmax(grid["enclosure"]))).clip(0,1)
grid["imageability"] = minmax(grid["bld_coverage"] ** 0.5)  # simple uniqueness proxy


In [ ]:
grid["PQI"] = (
    0.25 * grid["greenness"] +
    0.20 * grid["openness"] +
    0.20 * (1 - grid["enclosure"]) +
    0.20 * grid["walkability"] +
    0.15 * grid["imageability"]
).clip(0,1)


In [ ]:
# If you have existing sprawl/envdeg scores, combine them
if "sprawl_score" not in grid.columns:
    grid["sprawl_score"] = 1 - grid["PQI"]
if "envdeg_score" not in grid.columns:
    grid["envdeg_score"] = 1 - grid["greenness"]

grid["infra_deficiency"] = (1 - minmax(grid["road_density_km_per_km2"])) * (1 - grid["walkability"])
grid["combined_risk"] = (0.4*grid["sprawl_score"] + 0.3*grid["envdeg_score"] + 0.3*grid["infra_deficiency"]).clip(0,1)


In [ ]:
def recommend_v2(row):
    recs = []
    if row["combined_risk"] >= 0.6 and row["PQI"] < 0.4:
        recs += [
            "Prioritize green corridor development and canopy restoration.",
            "Retrofit streets for walkability and open sightlines.",
            "Add mixed-use nodes to enhance livability and reduce stress."
        ]
    elif row["infra_deficiency"] > 0.5:
        recs += [
            "Improve street connectivity and intersection density.",
            "Develop pedestrian and cycling infrastructure."
        ]
    elif row["envdeg_score"] > 0.6:
        recs += [
            "Expand green infrastructure (parks, riparian buffers).",
            "Encourage blue-green stormwater management solutions."
        ]
    if not recs:
        return "Monitor area; maintain current pattern."
    return "; ".join(dict.fromkeys(recs))

grid["recommendations_v2"] = grid.apply(recommend_v2, axis=1)


In [ ]:
import leafmap

geo_path = "/content/drive/MyDrive/planner/urban_perception_infra.geojson"
grid.to_file(geo_path, driver="GeoJSON")

m = leafmap.Map()
m.add_raster(TEST_RASTER, layer_name="NAIP Image")
m.add_geojson(geo_path, layer_name="Perceptual & Infra Diagnostics")
m


Map(center=[47.6464835, -117.59043650000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom…

In [ ]:
import geopandas as gpd
import rasterio, rasterio.mask
import numpy as np, pandas as pd
from shapely.geometry import box
from rasterstats import zonal_stats
import leafmap

# ---------- Configuration ----------
SEGMENTATION_RASTER = "/content/drive/MyDrive/planner/naip_test_semantic_prediction.tif"
OUT_GEOJSON = "/content/drive/MyDrive/planner/urban_perception_full.geojson"
OUT_GEOJSON_WGS84 = "/content/drive/MyDrive/planner/urban_perception_full_wgs84.geojson"
PIX_RES_FACTOR = 100   # each cell ~100× pixel size (tune for performance)

# ---------- Create analysis grid ----------
with rasterio.open(SEGMENTATION_RASTER) as src:
    bounds, res, crs = src.bounds, src.res[0], src.crs
    step = res * PIX_RES_FACTOR
    xmin, ymin, xmax, ymax = bounds.left, bounds.bottom, bounds.right, bounds.top

cells = [box(x, y, x+step, y+step)
         for x in np.arange(xmin, xmax, step)
         for y in np.arange(ymin, ymax, step)]
grid = gpd.GeoDataFrame(geometry=cells, crs=crs)

# ---------- Compute class ratios ----------
def class_ratio(array, cid):
    return np.count_nonzero(array == cid) / array.size if array.size > 0 else 0

stats = []
with rasterio.open(SEGMENTATION_RASTER) as src:
    for geom in grid.geometry:
        try:
            out, _ = rasterio.mask.mask(src, [geom], crop=True)
            arr = out[0]
            stats.append({
                "bld_coverage": class_ratio(arr, 1),
                "veg_coverage": class_ratio(arr, 2)
            })
        except Exception:
            stats.append({"bld_coverage": 0, "veg_coverage": 0})

grid = pd.concat([grid, pd.DataFrame(stats)], axis=1)

# ---------- Road density ----------
edges = edges.to_crs(grid.crs)
grid["road_density_km_per_km2"] = grid.geometry.apply(
    lambda g: edges.clip(g).length.sum()/1000/(g.area/1e6)
)

# ---------- Proxy indicators ----------
def minmax(x): return (x - x.min())/(x.max() - x.min() + 1e-9)

grid["greenness"] = minmax(grid["veg_coverage"])
grid["openness"] = 1 - grid["bld_coverage"]
grid["enclosure"] = grid["bld_coverage"]
grid["walkability"] = (minmax(grid["road_density_km_per_km2"]) *
                       (1 - minmax(grid["enclosure"]))).clip(0,1)
grid["imageability"] = minmax(np.sqrt(grid["bld_coverage"]))

# ---------- Perceptual Quality Index ----------
grid["PQI"] = (
    0.25*grid["greenness"] +
    0.20*grid["openness"] +
    0.20*(1 - grid["enclosure"]) +
    0.20*grid["walkability"] +
    0.15*grid["imageability"]
).clip(0,1)

# ---------- Composite risk ----------
grid["sprawl_score"] = 1 - grid["PQI"]
grid["envdeg_score"] = 1 - grid["greenness"]
grid["infra_deficiency"] = (1 - minmax(grid["road_density_km_per_km2"])) * (1 - grid["walkability"])
grid["combined_risk"] = (0.4*grid["sprawl_score"] +
                         0.3*grid["envdeg_score"] +
                         0.3*grid["infra_deficiency"]).clip(0,1)

# ---------- Management recommendations ----------
def recommend_v2(row):
    recs = []
    if row["combined_risk"] >= 0.6 and row["PQI"] < 0.4:
        recs += ["Green corridor & canopy restoration",
                 "Retrofit streets for walkability & sightlines",
                 "Add mixed-use nodes to enhance livability"]
    elif row["infra_deficiency"] > 0.5:
        recs += ["Improve street connectivity/intersections",
                 "Develop pedestrian & cycling infrastructure"]
    elif row["envdeg_score"] > 0.6:
        recs += ["Expand green infrastructure",
                 "Adopt blue–green stormwater systems"]
    return "; ".join(recs) if recs else "Monitor area; maintain current pattern."

grid["recommendations_v2"] = grid.apply(recommend_v2, axis=1)

# ---------- Aggregate overall results ----------
summary = {
    "mean_greenness": grid["greenness"].mean(),
    "mean_PQI": grid["PQI"].mean(),
    "mean_combined_risk": grid["combined_risk"].mean(),
    "high_risk_share": (grid["combined_risk"] > 0.6).sum() / len(grid)
}
summary_df = pd.DataFrame([summary])
print(" Overall summary metrics for full image:")
print(summary_df)

# ---------- Reproject to WGS84 for GeoJSON output ----------
grid_wgs84 = grid.to_crs(epsg=4326)
grid_wgs84.to_file(OUT_GEOJSON_WGS84, driver="GeoJSON")
print(f" Saved GeoJSON (WGS84) to {OUT_GEOJSON_WGS84}")

# ---------- Visualize ----------
m = leafmap.Map()
m.add_raster("/content/drive/MyDrive/planner/naip_test.tif", layer_name="Base Image")
m.add_geojson(OUT_GEOJSON_WGS84, layer_name="Perceptual & Risk Indicators")
m


 Overall summary metrics for full image:
   mean_greenness  mean_PQI  mean_combined_risk  high_risk_share
0             0.0  0.409964            0.836014              1.0
 Saved GeoJSON (WGS84) to /content/drive/MyDrive/planner/urban_perception_full_wgs84.geojson


Map(center=[47.6464835, -117.59043650000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom…

In [ ]:
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
import leafmap
from tempfile import NamedTemporaryFile

# ---------- Prepare raster ----------
bounds = grid_wgs84.total_bounds
res = 0.0001  # adjust resolution (~10 m at equator)
width = int((bounds[2] - bounds[0]) / res)
height = int((bounds[3] - bounds[1]) / res)
transform = from_bounds(*bounds, width=width, height=height)

# Rasterize PQI values
rasterized = rasterize(
    [(geom, val) for geom, val in zip(grid_wgs84.geometry, grid_wgs84["PQI"])],
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    dtype="float32"
)

# ---------- Save to temporary GeoTIFF with CRS ----------
with NamedTemporaryFile(suffix=".tif", delete=False) as tmp:
    tmp_tif = tmp.name
    with rasterio.open(
        tmp_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=rasterized.dtype,
        crs="EPSG:4326",
        transform=transform,
    ) as dst:
        dst.write(rasterized, 1)

print(f" Raster written to: {tmp_tif}")

# ---------- Display in Leafmap ----------
m = leafmap.Map()
m.add_raster(tmp_tif, layer_name="Perceptual Quality Index (PQI)", opacity=0.8, colormap="YlGn")

# ---------- Add colorbar ----------
cmap = cm.get_cmap("YlGn")
m.add_colorbar(
    title="Perceptual Quality Index (PQI)",
    colors=[colors.to_hex(cmap(i/10)) for i in range(11)],
    vmin=0,
    vmax=1,
    caption="Low → High"
)

m.zoom_to_bounds(bounds.tolist())
m


 Raster written to: /tmp/tmph85v1l8e.tif


In [ ]:
m


Map(center=[47.64662702176335, -117.59036594624568], controls=(ZoomControl(options=['position', 'zoom_in_text'…